<!--
Copyright (c) 2025 Milin Patel
Hochschule Kempten - University of Applied Sciences
Autonomous Driving: AI Safety and Security Workshop
MIT License - see LICENSE in the root directory.
-->


# Der KI-Blick — Wie sieht ein selbstfahrendes Auto?
### Campustage 2026 · *Automatisiertes Fahren! Sicher durch Zufall?*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/master/01_Perception_Systems/notebooks/00_workshop_der_ki_blick.ipynb)

**Milin Patel · Hochschule Kempten, IFM · MIT Licence**

---

Ein selbstfahrendes Auto "sieht" mit einer **Kamera** und einer **KI**.
In diesem Notebook schaust du der KI beim Sehen zu — und findest heraus, **wo sie versagt.**

So geht's: bei jeder Zelle links auf den **▶**-Knopf klicken (oder *Umschalt + Enter*).
Du musst **keinen Code lesen** — am Ende steuerst du alles über **Regler und Knöpfe.**


---
## Schritt 0 · Vorbereiten
Diese Zelle einmal ausführen. Sie lädt die KI (ca. 1 Minute).


In [ ]:
%pip install ultralytics -q
import io, numpy as np, requests
from PIL import Image, ImageDraw, ImageFilter
from ultralytics import YOLO
import ipywidgets as widgets
import matplotlib.pyplot as plt

try:
    from google.colab import output as _co
    _co.enable_custom_widget_manager()
except Exception:
    pass

ORANGE=(232,133,12); ROT=(200,60,60)
model = YOLO("yolov8n.pt")            # lädt sich selbst herunter
NAMES = model.names
DE = {"person":"Mensch","car":"Auto","bus":"Bus","truck":"LKW","bicycle":"Fahrrad",
      "motorcycle":"Motorrad","stop sign":"Stoppschild","traffic light":"Ampel"}
def de(n): return DE.get(n,n)

def lade_bild(url="https://ultralytics.com/images/bus.jpg"):
    r=requests.get(url,timeout=20); return Image.open(io.BytesIO(r.content)).convert("RGB")

BILD = lade_bild()
print("Bereit. Fahre mit der naechsten Zelle fort.")


---
## Schritt 1 · Du zählst zuerst

Hier ist eine echte Straßenszene. **Schau sie dir an und zähl mit den Augen:**

> **Wie viele Menschen siehst du? Und wie viele Fahrzeuge?**

Merk dir deine Zahl — gleich vergleichen wir sie mit der KI.


In [ ]:
plt.figure(figsize=(10,7)); plt.imshow(BILD); plt.axis("off")
plt.title("Zähl zuerst selbst — wie viele Menschen?"); plt.show()


---
## Schritt 2 · Jetzt schaut die KI

Ein Klick — die KI sucht. Jede **Box** ist etwas, das sie gefunden hat,
mit einer **Zahl**: wie sicher sie sich ist (die *Konfidenz*).

**Hat sie gleich viele gefunden wie du?**


In [ ]:
def erkenne(img, schwelle):
    res = model.predict(img, conf=schwelle, verbose=False)[0]
    out = img.copy(); d = ImageDraw.Draw(out); zeilen=[]
    for b in res.boxes:
        x1,y1,x2,y2 = [int(v) for v in b.xyxy[0]]
        cf=float(b.conf[0]); name=de(NAMES.get(int(b.cls[0]),""))
        col = ORANGE if cf>=0.5 else ROT
        d.rectangle([x1,y1,x2,y2], outline=col, width=3)
        d.text((x1+4, max(0,y1-12)), f"{name} {cf:.0%}", fill=col)
        zeilen.append(f"{name} {cf:.0%}")
    return out, zeilen

bild, gefunden = erkenne(BILD, 0.25)
plt.figure(figsize=(10,7)); plt.imshow(bild); plt.axis("off")
plt.title("Was die KI sieht — jede Box hat eine Zahl"); plt.show()
print("Gefunden:", "   ".join(gefunden) if gefunden else "nichts")


---
## Schritt 3 · Die Grenze entscheidet

Die KI ist sich **nie zu 100 % sicher** — sie gibt jeder Sache nur eine **Zahl**.
Ein Mensch legt fest, ab welcher Zahl etwas als *erkannt* gilt: die **Schwelle.**

Zieh den Regler:
- **Hoch** → nur noch sehr sichere Boxen bleiben. **Ein echter Mensch verschwindet.**
- **Runter** → die KI "sieht" plötzlich Dinge, die gar nicht da sind.

**Es gibt keine perfekte Einstellung. Und ein Mensch wählt sie.**


In [ ]:
def zeige_schwelle(Schwelle):
    plt.close("all")
    out, zeilen = erkenne(BILD, Schwelle)
    plt.figure(figsize=(10,7)); plt.imshow(out); plt.axis("off")
    plt.title(f"Schwelle = {Schwelle:.2f}   ({len(zeilen)} erkannt)"); plt.show()
    print("Erkannt:", "   ".join(zeilen) if zeilen else "NICHTS mehr.")

widgets.interact(zeige_schwelle,
    Schwelle=widgets.FloatSlider(min=0.05,max=0.95,step=0.05,value=0.25,continuous_update=False));


---
## Schritt 4 · Die Welt ist nicht sauber

Auf der Straße ist es nicht immer hell und klar. Wähl **Nebel** oder **Regen**
und schau, was mit der Erkennung passiert — die Zahlen fallen, Menschen verschwinden.

*Genau hier wird automatisiertes Fahren schwierig.*


In [ ]:
def wetter(img, art):
    if art=="Nebel":
        n=Image.new("RGB",img.size,(205,205,210))
        return Image.blend(img,n,0.55).filter(ImageFilter.GaussianBlur(2))
    if art=="Regen":
        o=img.copy(); d=ImageDraw.Draw(o); rng=np.random.default_rng(0)
        for _ in range(450):
            x=int(rng.integers(0,img.width)); y=int(rng.integers(0,img.height))
            d.line([(x,y),(x-2,y+12)], fill=(185,190,205))
        return Image.eval(o, lambda p:int(p*0.8))
    return img

def zeige_wetter(Wetter, Schwelle):
    plt.close("all")
    out, zeilen = erkenne(wetter(BILD, Wetter), Schwelle)
    plt.figure(figsize=(10,7)); plt.imshow(out); plt.axis("off")
    plt.title(f"Wetter: {Wetter}   |   Schwelle {Schwelle:.2f}   ({len(zeilen)} erkannt)"); plt.show()
    print("Erkannt:", "   ".join(zeilen) if zeilen else "NICHTS mehr.")

widgets.interact(zeige_wetter,
    Wetter=widgets.Dropdown(options=["Klar","Nebel","Regen"], value="Klar"),
    Schwelle=widgets.FloatSlider(min=0.05,max=0.95,step=0.05,value=0.25,continuous_update=False));


---
## Schritt 5 · (Freiwillig) Dein eigenes Bild

Willst du ein **eigenes** Foto testen — eine Straße, Autos, Menschen?
Führe die Zelle aus und wähl eine Bilddatei von deinem PC. Sonst überspring sie einfach.


In [ ]:
try:
    from google.colab import files
    hoch = files.upload()
    if hoch:
        name=list(hoch.keys())[0]
        BILD = Image.open(io.BytesIO(hoch[name])).convert("RGB")
        out,zeilen = erkenne(BILD,0.25)
        plt.figure(figsize=(10,7)); plt.imshow(out); plt.axis("off")
        plt.title("Dein Bild — was die KI darin sieht"); plt.show()
        print("Erkannt:", "   ".join(zeilen) if zeilen else "nichts")
except Exception:
    print("Kein Upload - kein Problem, mach einfach weiter.")


---
## Das nimmst du mit

> **Die KI sieht nicht, was du siehst. Sie sieht Zahlen — und Zahlen kann man täuschen.**

- Sie erkennt Dinge nur mit einer **Wahrscheinlichkeit**, nie sicher.
- Die **Schwelle** legt ein **Mensch** fest — zu hoch, echte Menschen verschwinden; zu niedrig, Fehlalarme.
- **Nebel, Regen oder ein Aufkleber** drücken die Zahl unter die Schwelle — die Person ist für die KI weg.

Das Problem ist **nicht gelöst.** Daran wird gearbeitet — vielleicht bald von dir.

---
*Mehr aus diesem Feld: die anderen Notebooks in diesem Repository (Perception, Failure Analysis, Cybersecurity, SOTIF ...) — für alle, die tiefer schauen wollen.*
